# Train the MINT contact head

Trains one inter-chain contact head with `contact/train_contact_head.py` and scores it on a held-out validation split.

1. **Data.** Two labelled files made by `contact/label_contacts.py`: `trainset_contacts.json` for training
   and `validset_contacts.json` for validation. Optionally take a random `N_TRAIN` / `N_VAL` clusters of each.
2. Extract embeddings for both.
3. Train the head.
4. Plot the training curves and report the best epoch.

**Runtime → Change runtime type → GPU** (T4 is enough).

`data/` is gitignored, so upload `interaction_head/data/trainset_contacts.json` and `validset_contacts.json`
to `TRAIN_CONTACTS` and `VAL_CONTACTS` on Drive first.
The branch in `BRANCH` must contain `contact/train_contact_head.py` and `extract_residue_embeddings.py`, so push them first.

Re-running is safe after a disconnect. Splits, embeddings and finished runs are saved on Drive and
reused, so each step only ever runs once. Each session copies the embeddings to the Colab disk before training, because reading them straight from Drive is slow.

In [ ]:
# ---- settings ----
DRIVE_DIR  = "/content/drive/MyDrive/mint"
CKPT       = f"{DRIVE_DIR}/mint.ckpt"               # cached so it's only downloaded once (~3.2 GB)
TRAIN_CONTACTS = f"{DRIVE_DIR}/trainset_contacts.json"  # label_contacts.py output; upload interaction_head/data/trainset_contacts.json here
VAL_CONTACTS   = f"{DRIVE_DIR}/validset_contacts.json"  # same for interaction_head/data/validset_contacts.json
WORK_DIR   = f"{DRIVE_DIR}/contact_head"            # subsets, embeddings, runs
SPLIT_DIR  = f"{WORK_DIR}/splits"                   # N_TRAIN / N_VAL subsets
EMB_DIR    = f"{WORK_DIR}/emb"                      # embeddings extracted here are saved and reused
RUNS_DIR   = f"{WORK_DIR}/runs"                     # one folder per training run
LOCAL_EMB  = "/content/emb"                         # training reads a local copy (much faster than Drive); None = read Drive directly

REPO_URL   = "https://github.com/AlecSilver/mint.git"
BRANCH     = "interaction_head"

# data
EXTRACT_LAYERS = [33]  # layers saved by extraction (33 = final layer)
N_TRAIN      = 2000    # random clusters of TRAIN_CONTACTS; None = all of them (embeddings for every pair take a lot of Drive space)
N_VAL        = None    # random clusters of VAL_CONTACTS; None = all of them
DATA_SEED    = 0       # which clusters are picked
CUTOFF       = 8.0     # Å, contact definition; must match the one both files were labelled with
MAX_TOKENS   = 16384   # extraction token budget per forward pass; lower it on CUDA out-of-memory
CROP         = "interface"  # extraction window for chains > 510 residues

# head training
LAYERS       = None    # subset of EXTRACT_LAYERS to train on; None = all of them (more than one -> learned layer mix)
PAIR         = "add"   # add | multiply
HIDDEN       = 128
BLOCKS       = 4
DROPOUT      = 0.1
EPOCHS       = 20
BATCH_SIZE   = 4
TRAIN_CROP   = 256
LR           = 1e-3
WEIGHT_DECAY = 0.01
POS_WEIGHT   = 1.0     # BCE weight on contacts (they are ~1% of cells)
DIST_WEIGHT  = 0.0     # weight of the distance loss; 0 = contacts only
ALL_PAIRS    = True    # every train pair each epoch; False = one per cluster (same thing when TRAIN_CONTACTS has one pair per cluster)
TRAIN_SEED   = 0
TRAIN_ARGS   = ""      # anything else for train_contact_head.py, e.g. "--workers 4"
RUN_NAME     = None    # run folder name; None = built from the settings, so changing them starts a new run

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(RUNS_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code and checkpoint

If the repo is private, use a URL with a personal access token
(`https://<token>@github.com/AlecSilver/mint.git`).

In [ ]:
REPO_DIR = "/content/mint"
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
HEAD_DIR = f"{REPO_DIR}/interaction_head"
for f in ["extract_residue_embeddings.py", "contact/train_contact_head.py"]:
    assert os.path.exists(f"{HEAD_DIR}/{f}"), f"{f} is missing on branch {BRANCH}; push it first"
for path in [TRAIN_CONTACTS, VAL_CONTACTS]:
    assert os.path.exists(path), f"{path} not found; upload interaction_head/data/{os.path.basename(path)} there"

# Colab already has torch, numpy, scipy, pandas and tqdm
!pip install -q --no-deps -e {REPO_DIR}

In [ ]:
if not os.path.exists(CKPT):
    !wget -q --show-progress -O {CKPT}.part https://huggingface.co/varunullanat2012/mint/resolve/main/mint.ckpt
    os.replace(CKPT + ".part", CKPT)

## 1. Data

Training uses `TRAIN_CONTACTS` and validation uses `VAL_CONTACTS`. When `N_TRAIN` / `N_VAL` is set, that
many clusters of the file are picked at random with `DATA_SEED` and saved as a subset. When it's `None`,
the whole file is used directly.

Subset names include the size and seed, so changing them makes new files instead of reusing stale ones.

In [ ]:
import json, random

SPLITS = ["train", "val"]
SOURCE = {"train": TRAIN_CONTACTS, "val": VAL_CONTACTS}
N_PICK = {"train": N_TRAIN, "val": N_VAL}
TAGS = {s: os.path.splitext(os.path.basename(SOURCE[s]))[0] + (f"_n{N_PICK[s]}_s{DATA_SEED}" if N_PICK[s] else "")
        for s in SPLITS}
SPLIT_FILE = {}

for split in SPLITS:
    SPLIT_FILE[split] = dst = f"{SPLIT_DIR}/{TAGS[split]}.json" if N_PICK[split] else SOURCE[split]
    if os.path.exists(dst) and dst != SOURCE[split]:
        print(f"{split}: exists: {dst}")
        continue
    by_cluster = {}
    with open(SOURCE[split]) as f:
        for line in f:
            rec = json.loads(line)
            assert rec["cutoff"] == CUTOFF, f"{SOURCE[split]} was labelled at {rec['cutoff']} Å, not CUTOFF = {CUTOFF}"
            by_cluster.setdefault(rec["cluster"], []).append(line)
    clusters = sorted(by_cluster)
    if dst == SOURCE[split]:
        print(f"{split}: all {sum(map(len, by_cluster.values()))} pairs in {len(clusters)} clusters of {dst}")
        continue
    random.Random(DATA_SEED).shuffle(clusters)
    picked = clusters[:N_PICK[split]]
    os.makedirs(SPLIT_DIR, exist_ok=True)
    with open(dst + ".part", "w") as f:
        for c in picked:
            f.writelines(by_cluster[c])
    os.replace(dst + ".part", dst)
    n = sum(len(by_cluster[c]) for c in picked)
    print(f"{split}: {n} pairs in {len(picked)} of {len(clusters)} clusters -> {dst}")

## 2. Embeddings

Extracts both splits to Drive. A split is skipped once it's complete, and resumes from the last
finished shard if it was cut off. The folder name includes the layers and crop mode, so changing either extracts a new set.

In [ ]:
LAYERS_STR = " ".join(map(str, sorted(EXTRACT_LAYERS)))
%cd {HEAD_DIR}
EMB = {}  # split -> embedding folder on Drive
for split in SPLITS:
    EMB[split] = out = f"{EMB_DIR}/{split}_{TAGS[split]}_L{'-'.join(map(str, sorted(EXTRACT_LAYERS)))}_{CROP}"
    if os.path.exists(f"{out}/DONE"):  # skip loading the 3.2 GB checkpoint just to find nothing to do
        print("exists:", out)
        continue
    !python extract_residue_embeddings.py {SPLIT_FILE[split]} {out} --checkpoint {CKPT} --max-tokens {MAX_TOKENS} --crop {CROP} --layers {LAYERS_STR}
    assert _exit_code == 0, f"extraction failed for {split}"
    open(f"{out}/DONE", "w").close()

# local copy for training; rsync only copies what's missing, so re-running is cheap
EMB_READ = dict(EMB)
if LOCAL_EMB:
    for split in SPLITS:
        !mkdir -p {LOCAL_EMB} && rsync -a --info=progress2 {EMB[split]} {LOCAL_EMB}/
        assert _exit_code == 0, f"copying {split} embeddings failed (disk full?)"
        EMB_READ[split] = f"{LOCAL_EMB}/{os.path.basename(EMB[split])}"
TRAIN_EMB, VAL_EMB = EMB_READ["train"], EMB_READ["val"]
!du -sh {TRAIN_EMB} {VAL_EMB}

## 3. Train

Writes `best.pt` (best validation AP), `last.pt` and `log.jsonl` to `RUN_DIR` on Drive. A finished run is
skipped when you re-run this cell. An unfinished one (cut off by a disconnect) is restarted from scratch,
because the script can't resume. Unless `RUN_NAME` is set, the folder name includes a hash of the settings,
so changing any of them starts a new run instead of overwriting the old one.

In [ ]:
import hashlib, shutil

layers = sorted(LAYERS or EXTRACT_LAYERS)
assert set(layers) <= set(EXTRACT_LAYERS), f"LAYERS {layers} not all in EXTRACT_LAYERS {EXTRACT_LAYERS}"
settings = dict(layers=layers, pair=PAIR, hidden=HIDDEN, blocks=BLOCKS, dropout=DROPOUT, epochs=EPOCHS,
                batch=BATCH_SIZE, crop=TRAIN_CROP, lr=LR, weight_decay=WEIGHT_DECAY, pos_weight=POS_WEIGHT,
                dist_weight=DIST_WEIGHT, all_pairs=ALL_PAIRS, seed=TRAIN_SEED, cutoff=CUTOFF, extra=TRAIN_ARGS)
RUN_DIR = f"{RUNS_DIR}/" + (RUN_NAME or f"{TAGS['train']}_L{'-'.join(map(str, layers))}_{PAIR}_h{HIDDEN}_e{EPOCHS}_" +
                            hashlib.md5(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:6])
LOG = f"{RUN_DIR}/log.jsonl"

def finished():
    return os.path.exists(LOG) and sum(1 for _ in open(LOG)) >= EPOCHS

%cd {HEAD_DIR}
if finished():
    print("done, skipping:", RUN_DIR)
else:
    shutil.rmtree(RUN_DIR, ignore_errors=True)  # log.jsonl appends, so never continue a partial run
    os.makedirs(RUN_DIR)
    with open(f"{RUN_DIR}/settings.json", "w") as f:
        json.dump(settings, f, indent=2)
    LAYERS_ARG = " ".join(map(str, layers))
    ALL_PAIRS_ARG = "--all-pairs" if ALL_PAIRS else ""
    !python contact/train_contact_head.py --train-emb {TRAIN_EMB} --train-contacts {SPLIT_FILE['train']} --val-emb {VAL_EMB} --val-contacts {SPLIT_FILE['val']} --out {RUN_DIR} --layers {LAYERS_ARG} --pair {PAIR} --hidden {HIDDEN} --blocks {BLOCKS} --dropout {DROPOUT} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --crop {TRAIN_CROP} --lr {LR} --weight-decay {WEIGHT_DECAY} --pos-weight {POS_WEIGHT} --dist-weight {DIST_WEIGHT} --cutoff {CUTOFF} --seed {TRAIN_SEED} {ALL_PAIRS_ARG} {TRAIN_ARGS}
    assert _exit_code == 0, "training failed"

## 4. Results

The best epoch is the one with the highest validation AP (the one saved as `best.pt`). AP is computed over all
validation cells. Top-L/5 is the share of each pair's L/5 highest-scoring cells that are true contacts (L = shorter chain).

In [ ]:
import pandas as pd

log = pd.DataFrame([json.loads(l) for l in open(LOG)])
best = log.loc[log.val_ap.idxmax()]
print(f"best epoch {int(best.epoch)} of {len(log)}: val AP {best.val_ap:.4f}, top10 {best.val_top10:.4f}, "
      f"top-L/10 {best['val_topL/10']:.4f}, top-L/5 {best['val_topL/5']:.4f}")
print("positive rate (AP of a random guess):", round(best.val_pos_rate, 4))
if len(layers) > 1:
    print("layer weights:", {l: round(w, 3) for l, w in zip(layers, best.layer_weights)})
print("checkpoint:", f"{RUN_DIR}/best.pt")
log.drop(columns=["layer_weights"]).round(4)

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID = "#1f2328", "#59636e", "#d8dee4"
TRAIN_C, VAL_C = "#2f6fdf", "#e3742f"
x = log.epoch + 1

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [("Loss", [("train_loss", "train", TRAIN_C), ("val_loss", "val", VAL_C)]),
          ("Validation AP", [("val_ap", None, VAL_C)]),
          ("Validation top-L/5 precision", [("val_topL/5", None, VAL_C)])]
for ax, (title, series) in zip(axes, panels):
    for col, label, color in series:
        ax.plot(x, log[col], color=color, lw=2, marker="o", ms=5, mec="white", mew=1.2, zorder=3)
        if label:  # direct labels instead of a legend
            ax.annotate(label, (x.iloc[-1], log[col].iloc[-1]), textcoords="offset points", xytext=(6, 0),
                        va="center", color=color, fontsize=9)
    ax.axvline(best.epoch + 1, color=MUTED, lw=1, ls="--", zorder=2)
    ax.set_title(title, loc="left", color=INK, fontsize=11, pad=12)
    ax.set_xlabel("epoch", color=MUTED)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.tick_params(colors=MUTED, length=0)
    for s in ax.spines.values():
        s.set_visible(False)
axes[1].annotate(f"best {best.val_ap:.3f}", (best.epoch + 1, best.val_ap), textcoords="offset points",
                 xytext=(0, 9), ha="center", color=INK, fontsize=9)
axes[1].margins(y=0.2)
fig.tight_layout()
fig.savefig(f"{RUN_DIR}/curves.png", dpi=150)
plt.show()
print("saved to", RUN_DIR)